# Certify the fault distance of circuits

Brute-force fault-distance enumeration is expensive because the number of fault combinations grows rapidly.
`circuits.get_fault_distance` instead converts a noisy Stim circuit into an auxiliary CSS code and uses the Brouwer-Zimmermann code-distance algorithm to find the fewest faults that flip an observable without firing a detector.
See `help(circuits.get_fault_distance)` for the construction.

The examples below certify a surface-code memory experiment and a flagged Steane-state preparation circuit.

## Setup

Run the next cell once in a fresh notebook environment.
IPython's `%pip` magic installs into the active kernel, so the following imports use the same environment.

In [1]:
%%capture
%pip install qldpc

In [2]:
import stim

from qldpc import circuits, codes

## Certify the fault distance of a memory experiment

We test X- and Z-basis memory of a distance-5 surface code with five rounds of syndrome extraction.
Stim's graphlike search provides a cross-check here, because every fault in this circuit flips at most two detectors, whereas `circuits.get_fault_distance` does not require graphlike faults.

In [3]:
code = codes.SurfaceCode(5)
noise_model = circuits.DepolarizingNoiseModel(1e-3)

for basis in ("X", "Z"):
    # one logical memory cycle = code.get_distance() rounds of syndrome extraction
    # you can replace logical_memory_cycle with another memory circuit
    initialization, logical_memory_cycle, readout, *_ = circuits.get_memory_experiment_parts(
        code, basis=basis, num_rounds=code.get_distance()
    )

    # keep initialization and readout noiseless, to certify only the inner QEC cycle
    noisy_circuit = initialization + noise_model.noisy_circuit(logical_memory_cycle) + readout
    fault_distance = circuits.get_fault_distance(noisy_circuit)
    graphlike_distance = len(noisy_circuit.shortest_graphlike_error())
    assert fault_distance == graphlike_distance
    print(
        f"{basis}-basis memory: fault distance {fault_distance}, "
        f"Stim graphlike distance {graphlike_distance}"
    )

X-basis memory: fault distance 5, Stim graphlike distance 5
Z-basis memory: fault distance 5, Stim graphlike distance 5


## Certify the fault distance of a state-preparation circuit

We prepare the Steane code in $|0_L\rangle$ with the circuit from the [state-preparation notebook](5_state_preparation.ipynb).
Its final X-basis measurement of qubit 7 flags logical-X faults, and a nonzero outcome rejects the attempt.

In [4]:
code = codes.SteaneCode()

state_prep_circuit = stim.Circuit("""
    # non-fault-tolerant state preparation
    H 0 2 4
    CX 0 3 2 1 4 5
    CX 0 1 2 6 4 3
    CX 2 5 3 6
    # flag errors in a logical Z representative
    H 7
    CZ 7 1 7 3 7 5
    MX 7
""")

`circuits.get_state_prep_diagnostic_circuit` appends a flag detector for the preparation measurement, noiseless final stabilizer measurements, and a noiseless logical-Z observable that it infers from the prepared state.
The fault distance is then the fewest faults that flip this observable while the flag and the final stabilizer syndrome both stay zero.

In [5]:
diagnostic_circuit, _ = circuits.get_state_prep_diagnostic_circuit(
    code,
    state_prep_circuit,
    add_flags=True,
)
noisy_circuit = noise_model.noisy_circuit(diagnostic_circuit)
fault_distance = circuits.get_fault_distance(noisy_circuit)
print(f"State-preparation undetected logical fault distance: {fault_distance}")

State-preparation undetected logical fault distance: 3
